# RNA 3D Structure Prediction — Kaggle TPU v5e-8 Training

Trains the full MSA Transformer + Geometric Structure Module on **all 5 716 training sequences** using all **8 TPU chips** in parallel (bfloat16, XMP data-parallel).

**Before running:**
1. Enable TPU in *Notebook settings → Accelerator → TPU v5e-8* (or TPU v3-8)
2. Add the competition dataset via *+ Add Data → `stanford-rna-3d-folding`*
3. (Optional) Fork or make public if the GitHub repo is private.

**What this notebook does:**
- Clones the training code from GitHub
- Auto-finds the competition CSV/MSA files and symlinks them
- Runs `train_tpu.py`, which spawns one process per TPU chip via `torch_xla XMP`
- Each chip trains on its own data shard in **bfloat16** (native TPU precision)
- Checkpoints + history are zipped for download

Estimated runtime on TPU v5e-8: ~2–3 h for 50 epochs × 5 716 sequences.


In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')

# Check TPU presence WITHOUT calling xm.xla_device().
# IMPORTANT: calling xm.xla_device() here initialises the XLA computation
# client in single-process (local) mode.  When torch_xla.launch later tries
# to spawn 8 parallel PJRT workers it fails with
#   "InitializeComputationClient() can only be called once"  or
#   "Expected 8 worker addresses, got 1"
# So we only inspect environment variables — no XLA API calls.
try:
    import torch_xla  # noqa: F401 — just verify the package is importable
    tpu_env = os.environ.get('KAGGLE_TPU_ADDR', os.environ.get('TPU_NAME', ''))
    pjrt_dev = os.environ.get('PJRT_DEVICE', '')
    print(f'TPU      : package OK  (PJRT_DEVICE={pjrt_dev!r}, TPU_NAME/ADDR={tpu_env!r})')
    print('TPU chips: will be detected inside train_tpu.py worker processes')
except Exception as e:
    print(f'TPU      : NOT detected — {e}')
    print('WARNING: Switch Notebook settings → Accelerator → TPU v5e-8')

# Check GPU (fallback)
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM)')


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'biopython>=1.81', 'einops>=0.6.1', 'tqdm>=4.65.0'],
    check=True
)
print('Dependencies ready.')

In [ ]:
import os, subprocess

REPO   = 'https://github.com/shanmugavalli/rna-structure.git'
WORKDIR = '/kaggle/working/rna-structure'

if os.path.exists(f'{WORKDIR}/.git'):
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
print(f'Working directory: {os.getcwd()}')

for f in ['src/train.py', 'src/config.py', 'src/model.py', 'src/dataset.py']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

In [ ]:
import os

def _find_competition_input():
    """Scan /kaggle/input up to 2 levels deep for a folder with train_sequences.csv."""
    marker = 'train_sequences.csv'
    # Explicit known paths first (fastest)
    explicit = [
        '/kaggle/input/competitions/stanford-rna-3d-folding-2',
        '/kaggle/input/competitions/stanford-rna-3d-folding',
        '/kaggle/input/stanford-rna-3d-folding-2',
        '/kaggle/input/stanford-rna-3d-folding',
    ]
    for p in explicit:
        if os.path.isdir(p) and os.path.isfile(os.path.join(p, marker)):
            return p
    # Fallback: scan 2 levels deep
    for name in os.listdir('/kaggle/input/'):
        p = f'/kaggle/input/{name}'
        if not os.path.isdir(p):
            continue
        if os.path.isfile(os.path.join(p, marker)):
            return p
        try:
            for sub in os.listdir(p):
                pp = os.path.join(p, sub)
                if os.path.isdir(pp) and os.path.isfile(os.path.join(pp, marker)):
                    return pp
        except PermissionError:
            pass
    return None

COMPETITION_INPUT = _find_competition_input()

if COMPETITION_INPUT is None:
    print('Could not find competition data. Available inputs:')
    for name in os.listdir('/kaggle/input/'):
        p = f'/kaggle/input/{name}'
        print(f'  {p}/')
        try:
            for f in sorted(os.listdir(p))[:8]:
                print(f'    {f}')
        except Exception:
            pass
    raise FileNotFoundError(
        'Add the competition dataset via "+ Add Data" in the Kaggle sidebar '
        '(search for "stanford-rna-3d-folding").'
    )

print(f'Competition data found at: {COMPETITION_INPUT}')

# Create output directories
for d in ['data/raw', 'data/processed', 'outputs/checkpoints',
          'outputs/logs', 'outputs/predictions', 'outputs/analysis']:
    os.makedirs(d, exist_ok=True)

# Symlink CSV files so the training code finds them at expected paths
csv_files = [
    'train_sequences.csv', 'train_labels.csv',
    'validation_sequences.csv', 'validation_labels.csv',
    'test_sequences.csv', 'sample_submission.csv',
]
for fname in csv_files:
    src = os.path.join(COMPETITION_INPUT, fname)
    dst = os.path.join('data/raw', fname)
    if os.path.isfile(src) and not os.path.lexists(dst):
        os.symlink(src, dst)
        print(f'  linked: {fname}')
    elif not os.path.isfile(src):
        print(f'  WARNING: {fname} not found — skipping')

# Symlink MSA directory
msa_src = os.path.join(COMPETITION_INPUT, 'MSA')
msa_dst = 'data/raw/MSA'
if os.path.isdir(msa_src):
    if not os.path.lexists(msa_dst):
        os.symlink(msa_src, msa_dst)
        n_msa = len(os.listdir(msa_src))
        print(f'  linked: MSA/ ({n_msa} files)')
else:
    print('  WARNING: MSA/ folder not found — MSA features will be disabled')
    os.makedirs(msa_dst, exist_ok=True)

print('Data setup complete.')


## Training Configuration (TPU v5e-8)

| Setting | Value | Notes |
|---|---|---|
| `RNA_RUNTIME` | `tpu` | selects `full` model + XLA device |
| `RNA_EPOCHS` | `50` | ~2–3 h on v5e-8 |
| `RNA_LR` | `3e-4` | higher LR works on TPU (large eff. batch) |
| `RNA_BATCH_SIZE` | `4` | per chip → 4 × 8 chips = **32 global** |
| `RNA_MAX_SEQ_LENGTH` | `384` | fixed shape → zero XLA recompilations |
| `RNA_MAX_MSA_SEQS` | `8` | 8 MSA rows per sequence |
| `RNA_MSA_DEPTH` | `4` | 4 Evoformer blocks (larger than GPU default) |
| `RNA_STRUCTURE_ITERATIONS` | `3` | 3 refinement cycles |
| `RNA_AUG_ROT` | `0.8` | rotation augmentation (applied on CPU workers) |
| `RNA_AUG_NOISE` | `0.3` | coordinate noise augmentation |
| `RNA_MAX_TRAIN_MINUTES` | `210` | hard stop 15 min before Kaggle kills kernel |

Change `extra_env` below to tune any setting before re-running.


In [ ]:
import os, subprocess, sys

extra_env = {
    'RNA_RUNTIME'              : 'tpu',
    'RNA_EPOCHS'               : '50',
    'RNA_LR'                   : '3e-4',
    'RNA_BATCH_SIZE'           : '4',
    'RNA_MAX_SEQ_LENGTH'       : '384',
    'RNA_MAX_MSA_SEQS'         : '8',
    'RNA_MSA_DEPTH'            : '4',
    'RNA_STRUCTURE_ITERATIONS' : '3',
    'RNA_AUG_ROT'              : '0.8',
    'RNA_AUG_NOISE'            : '0.3',
    'RNA_AUG_NOISE_STD'        : '0.1',
    'RNA_VAL_FREQUENCY'        : '2',
    # Hard-stop 15 min before the 4-hour kernel deadline
    'RNA_MAX_TRAIN_MINUTES'    : '210',
}

env = {**os.environ, **extra_env}
LOG = 'outputs/logs/kaggle_gpu_train.log'
os.makedirs('outputs/logs', exist_ok=True)

print('Starting TPU training via train_tpu.py ...')
print(f'Key settings: {extra_env}')
print('-' * 70)

with open(LOG, 'w') as log_f:
    proc = subprocess.Popen(
        [sys.executable, '-u', 'src/train_tpu.py'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
    proc.wait()

print(f'\nExit code : {proc.returncode}')
print(f'Log saved : {LOG}')


In [ ]:
import re, glob, os

LOG = 'outputs/logs/kaggle_gpu_train.log'

# --- Parse TM scores from log ---
tm_history = []
with open(LOG) as f:
    for line in f:
        m = re.search(r'\[Epoch\s+(\d+)\].*Val TM:\s+([0-9.]+)', line)
        if not m:
            m = re.search(r'Val TM:\s+([0-9.]+)', line)
            if m:
                tm_history.append(float(m.group(1)))
        else:
            tm_history.append(float(m.group(2)))

if tm_history:
    best = max(tm_history)
    last = tm_history[-1]
    print(f'Epochs recorded : {len(tm_history)}')
    print(f'Best Val TM     : {best:.4f}')
    print(f'Last Val TM     : {last:.4f}')
else:
    print('No Val TM lines found in log — check log for errors.')

# --- Last 30 lines of log ---
print('\n--- Last 30 log lines ---')
with open(LOG) as f:
    tail = f.readlines()[-30:]
print(''.join(tail))

# --- List checkpoints ---
ckpts = sorted(glob.glob('outputs/checkpoints/*.pt'))
print(f'\nCheckpoints saved ({len(ckpts)}):')
for c in ckpts:
    print(f'  {os.path.basename(c)}  ({os.path.getsize(c)/1e6:.1f} MB)')

In [ ]:
import shutil, os

# Zip log + checkpoints for easy download from Kaggle outputs panel
shutil.make_archive('/kaggle/working/training_results', 'zip', 'outputs')
size = os.path.getsize('/kaggle/working/training_results.zip') / 1e6
print(f'training_results.zip created: {size:.1f} MB')
print('Download from the Kaggle output panel on the right.')